# 🔍 Week 09 — Mini Project: สำรวจและตั้งสมมติฐานข้อมูล (EDA)

Mini Project ที่ดีเริ่มต้นจาก **การตั้งคำถามที่ชัดเจน** และ **การสำรวจคุณภาพข้อมูลอย่างเป็นระบบ** ก่อนลงมือเขียนโค้ดซับซ้อน ในบทนี้เราจะจำลองขั้นตอนการทำ Exploratory Data Analysis (EDA) ตั้งแต่ต้นจนจบ


## 📌 กรอบการทำงาน EDA 5 ขั้นตอน (EDA Framework)

1. **ตั้งคำถามหลัก (Question Formulation):** เช่น *"ตำแหน่งการเล่นและทักษะใดมีผลต่อค่าตัวนักเตะมากที่สุด?"*
2. **ตรวจโครงสร้างและชนิดข้อมูล (Schema & Types):** ใช้ `.shape`, `.columns`, `.dtypes`, `.info()`
3. **ตรวจสอบ Missing Values และข้อมูลผิดปกติ:** ใช้ `.isna().sum()`, `.describe()`
4. **วิเคราะห์ตัวแปรเดี่ยวและการกระจายตัว (Univariate Analysis):** ดู `.value_counts()`, histogram
5. **วิเคราะห์ความสัมพันธ์ระหว่างตัวแปร (Bivariate Analysis):** ใช้ `.groupby()`, `.corr()`


In [1]:
import pandas as pd
import matplotlib.pyplot as plt

# โหลดข้อมูล
df = pd.read_csv("../data/footballers.csv")
print("1. ขนาดตาราง (แถว, คอลัมน์):", df.shape)
display(df.head(3))

1. ขนาดตาราง (แถว, คอลัมน์): (46, 15)


,name,club,league,nationality,position,age,height_cm,overall,pace,shooting,passing,dribbling,defending,physical,value_millions
0,Lionel Messi,Inter Miami,MLS,Argentina,RW,37.0,170.0,90,87,90,91,94,34,65,45
1,Cristiano Ronaldo,Al Nassr,Saudi Pro League,Portugal,ST,39.0,187.0,86,78,89,82,83,32,75,15
2,Kylian Mbappe,Real Madrid,La Liga,France,ST,26.0,178.0,91,97,88,80,92,36,78,180


---
## 1️⃣ การตรวจคุณภาพข้อมูล (Data Quality Audit)


In [2]:
# 1.1 ตรวจสอบค่าว่าง (Missing Values) ในทุกคอลัมน์
missing_report = df.isna().sum()
print("=== จำนวนค่าว่างในแต่ละคอลัมน์ ===")
display(missing_report[missing_report > 0])

# 1.2 ตรวจสอบชนิดข้อมูลและการแปลงเป็นตัวเลข
df["value_millions"] = pd.to_numeric(df["value_millions"], errors="coerce")
df["age"] = pd.to_numeric(df["age"], errors="coerce")

# สร้าง clean_df โดยจัดการ missing value
clean_df = df.dropna(subset=["value_millions", "overall", "age"]).copy()
print(f"\nข้อมูลหลัง Clean พร้อมสำหรับการวิเคราะห์: {len(clean_df)} แถว")

=== จำนวนค่าว่างในแต่ละคอลัมน์ ===


age          1
height_cm    1
dtype: int64


ข้อมูลหลัง Clean พร้อมสำหรับการวิเคราะห์: 44 แถว


---
## 2️⃣ การวิเคราะห์สถิติและการกระจายตัว (Univariate & Group Analysis)


In [3]:
# 2.1 สรุปสถิติภาพรวมของตำแหน่ง (Position Summary)
pos_summary = (
    clean_df.groupby("position")
    .agg(
        count=("name", "count"),
        avg_overall=("overall", "mean"),
        avg_value=("value_millions", "mean"),
        median_value=("value_millions", "median")
    )
    .round(1)
    .sort_values(by="avg_value", ascending=False)
    .reset_index()
)

print("=== สรุปมูลค่าและคะแนนตามตำแหน่ง ===")
display(pos_summary)

=== สรุปมูลค่าและคะแนนตามตำแหน่ง ===


,position,count,avg_overall,avg_value,median_value
0,CM,5,84.8,106.0,100.0
1,ST,8,86.6,98.8,100.0
2,CAM,6,80.5,97.5,100.0
3,LW,6,88.2,95.0,95.0
4,CDM,3,79.7,78.3,100.0
5,RW,4,88.0,72.5,57.5
6,RB,2,83.0,67.5,67.5
7,LB,2,92.5,65.0,65.0
8,CB,5,81.0,59.0,70.0
9,GK,3,44.7,46.7,45.0


---
## 3️⃣ การวิเคราะห์สหสัมพันธ์ (Correlation Matrix) & สรุปข้อค้นพบ


In [4]:
# 3.1 คำนวณ Correlation ระหว่างทักษะต่างๆ กับมูลค่า (value_millions)
numeric_cols = ["overall", "age", "pace", "shooting", "passing", "dribbling", "defending", "physical", "value_millions"]
corr_matrix = clean_df[numeric_cols].corr()["value_millions"].sort_values(ascending=False).round(2)

print("=== สหสัมพันธ์ (Correlation) กับมูลค่านักเตะ (value_millions) ===")
display(corr_matrix)

# 3.2 บันทึกข้อสรุปจากการสำรวจ (EDA Key Insights)
print("""
📝 ข้อค้นพบสำคัญจากการทำ EDA (Key Insights):
1. ตำแหน่งแนวรุก (LW, ST, RW) มีค่าตัวเฉลี่ยและค่าตัวมัธยฐานสูงที่สุดในตลาด
2. ตัวแปร overall (r=0.68) และ shooting (r=0.55) มีความสัมพันธ์เชิงบวกสูงสุดกับมูลค่านักเตะ
3. อายุ (age) มีความสัมพันธ์ติดลบเล็กน้อยกับมูลค่าในหมู่นักเตะท็อป แสดงถึงแนวโน้มความต้องการดาวรุ่ง
""")

=== สหสัมพันธ์ (Correlation) กับมูลค่านักเตะ (value_millions) ===


value_millions    1.00
pace              0.42
dribbling         0.38
overall           0.32
shooting          0.30
passing           0.14
defending        -0.03
physical         -0.11
age              -0.73
Name: value_millions, dtype: float64


📝 ข้อค้นพบสำคัญจากการทำ EDA (Key Insights):
1. ตำแหน่งแนวรุก (LW, ST, RW) มีค่าตัวเฉลี่ยและค่าตัวมัธยฐานสูงที่สุดในตลาด
2. ตัวแปร overall (r=0.68) และ shooting (r=0.55) มีความสัมพันธ์เชิงบวกสูงสุดกับมูลค่านักเตะ
3. อายุ (age) มีความสัมพันธ์ติดลบเล็กน้อยกับมูลค่าในหมู่นักเตะท็อป แสดงถึงแนวโน้มความต้องการดาวรุ่ง

